# 01.5 Gaussian KDE VAF and Alt.Depth plots with sample-frequency filter

This notebook calculates sample frequency across samples, removes variants above a sample-frequency cutoff, then plots VAF and AltDepth using explicit Gaussian KDE.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

run_label = "Run2"
input_csv = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/04_qc_checking_on_target/04_Run2_per_variant_target_status_FULL.csv")
#output_dir = Path("/home/donetski/Notebooks/OutputFiles/01.5_GAUSSIAN_vaf_density_graphs") / run_label.lower()
output_dir = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/01.5_GAUSSIAN_vaf_density_graphs") / run_label.lower()
output_dir.mkdir(parents=True, exist_ok=True)

output_prefix = "01.5"

filter_to_pass = True
filter_to_on_target = True
callers = ["DeepVariant", "Mutect2"]
variant_cols = ["Chr", "Start", "REF", "ALT"]

# Remove variants seen in more than this fraction of samples.
sample_frequency_thresholds = [0.10
                              ]
#Sample.AltDepth >= threshold
altdepth_thresholds = [1, 3, 5, 7, 10]
vaf_thresholds = [0.01, 0.03, 0.05, 0.07, 0.10]

TARGET_GENES = [
    "ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1",
    "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53",
]

# Optional safety cap so gaussian_kde does not run on millions of points.
# Set to None if you want to use all points.
#max_kde_points = 100_000

In [ ]:
df = pd.read_csv(input_csv, low_memory=False, encoding="latin1")

required_columns = ["Sample.ID", "caller", "Gene", "Sample.AltFrac", "Sample.AltDepth"] + variant_cols
df["Sample.AltFrac"] = pd.to_numeric(df["Sample.AltFrac"], errors="coerce")
df["Sample.AltDepth"] = pd.to_numeric(df["Sample.AltDepth"], errors="coerce")

## 1. Calculate sample frequency

A variant is counted once per sample using exact variant identity: `Chr`, `Start`, `REF`, `ALT`. This prevents the same sample-variant from being counted twice if both callers found it.

In [ ]:
n_samples = df["Sample.ID"].nunique()

# One row per sample + exact variant.
one_row_per_sample_variant = df.drop_duplicates(subset=["Sample.ID"] + variant_cols).copy()

frequency_df = (
    one_row_per_sample_variant
    .groupby(variant_cols)
    .agg(sample_count=("Sample.ID", "nunique"))
    .reset_index()
)

frequency_df["total_samples_in_run"] = n_samples
frequency_df["sample_fraction"] = frequency_df["sample_count"] / n_samples
frequency_df["sample_percent"] = frequency_df["sample_fraction"] * 100

# Add frequency columns back onto every original row.
df = df.merge(frequency_df, on=variant_cols, how="left")

with_frequency_csv = output_dir / f"{output_prefix}_{run_label}_with_sample_frequency.csv"
df.to_csv(with_frequency_csv, index=False)

print("Saved:", with_frequency_csv)
df[["Sample.ID", "caller", "Gene"] + variant_cols + ["sample_count", "total_samples_in_run", "sample_fraction"]].head()

## 2. Small helper functions

`plot_gaussian_kde` is the main plotting helper. It explicitly uses `gaussian_kde`, so both VAF and AltDepth are smoothed the same way.

In [ ]:
def safe_name(value):
    return str(value).replace(" ", "_").replace("/", "_").replace(".", "p").lower()

def threshold_label(value):
    if value is None:
        return "none"
    return str(value).replace(".", "p")

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.show()
    print("Saved:", path)

def plot_gaussian_kde(series, label):
    values = pd.to_numeric(series, errors="coerce").dropna()

    # KDE requires at least two non-identical values.
    if len(values) < 2 or values.nunique() < 2:
        return False

    # Fit KDE using every valid value.
    kde = gaussian_kde(values)

    # Evaluate KDE across the complete observed range.
    x_grid = np.linspace(values.min(), values.max(), 300)

    plt.plot(x_grid, kde(x_grid), label=label)
    return True

## 3. Make analysis sets

The same sample-frequency values are used for both sets. The 16-gene set is just a subset for plotting.

In [ ]:
target_gene_mask = df["Gene"].astype(str).str.upper().isin(TARGET_GENES)

analysis_sets = {
    "all_variants": df.copy(),
    "target_16_genes": df[target_gene_mask].copy(),
}

for set_name, set_df in analysis_sets.items():
    print(set_name, "rows:", f"{len(set_df):,}")

## 4. Filter by sample frequency and make KDE plots

For each caller and analysis set, this saves:

1. VAF Gaussian KDE by AltDepth threshold.
2. AltDepth Gaussian KDE by AltDepth threshold.
3. AltDepth Gaussian KDE before vs after sample-frequency filtering, with no AltDepth cutoff.